# Debate: Adversarial Multi-Agent Reasoning

| Property | Value |
|---|---|
| Origin | Du et al., *Improving Factuality and Reasoning in Language Models through Multiagent Debate* (2023). [arXiv:2305.14325](https://arxiv.org/abs/2305.14325) |

## Definition

**Debate** is a multi-agent pattern in which two or more agents are assigned *opposing stances* on a genuinely debatable question and argue their side across several rounds, each rebutting the other's most recent point after seeing the full transcript so far. A neutral **Judge** agent then reviews the entire exchange and synthesizes a final answer that weighs the strongest points from each side.

The pattern is inspired by the `28_debate` concept in FareedKhan-dev's `all-agentic-architectures` collection: instead of asking a single LLM call for "the answer", you force the model to argue *both* sides of a hard question, then let a third, unaligned agent adjudicate. The adversarial pressure surfaces considerations a single pass would likely omit or under-weight.

## High-Level Workflow

```
                 ┌────────────────────┐
        START ──▶│  Debater A (Pro)    │──┐
                 └────────────────────┘  │
                           ▲              │  full transcript so far
                           │              ▼
                 ┌────────────────────┐
                 │  Debater B (Con)    │
                 └────────────────────┘
                           │
                 (repeat for N rounds)
                           │
                           ▼
                 ┌────────────────────┐
                 │   Judge Agent        │──▶ END (structured verdict)
                 └────────────────────┘
```

Each debater node reads the *entire* transcript accumulated so far (not just the opponent's last turn) so it can maintain consistency with its own earlier arguments while directly rebutting the newest opposing point. A round counter drives a conditional edge that loops the two debaters back and forth until `max_rounds` is reached, then routes to the judge.

## When to Use

- Questions with **legitimate trade-offs** and no single objectively correct answer (architecture decisions, policy questions, build-vs-buy, strategy choices).
- When a single-agent answer tends to anchor on the first framing it generates and under-explore the counter-case.
- When you want an auditable trail showing *why* a recommendation was made, including the strongest counter-arguments that were considered and rejected.

## Strengths

- Forces explicit consideration of counter-arguments instead of one-sided reasoning.
- The judge's synthesis is typically more nuanced and better-hedged than either debater's own conclusion.
- Transparent: the full transcript is a readable audit trail of the reasoning process.

## Weaknesses

- More expensive: `2N + 1` LLM calls instead of 1.
- Debaters can still both miss a consideration neither persona was primed to raise.
- Quality depends heavily on persona/system-prompt design — a weak "Con" persona produces a lopsided, unhelpful debate.
- The judge itself is a single point of failure/bias unless it is deliberately prompted to be even-handed.

In [ ]:
# ============ SETUP ============
from dotenv import load_dotenv

load_dotenv()

In [ ]:
# ============ IMPORTS ============
from typing import TypedDict, Literal, List
from pydantic import BaseModel, Field
from langchain_core.messages import SystemMessage, HumanMessage
from langgraph.graph import END, START, StateGraph

from helpers import get_llm

llm = get_llm()

## What We Are Going to Do

1. Pick a genuinely debatable, technical trade-off question: *"Should this team adopt a microservices architecture or a monolith for their new product?"*
2. Define two debaters with distinct assigned stances:
   - **Pro-Microservices Advocate** — argues for microservices.
   - **Pro-Monolith Advocate** — argues for a monolith.

   Each is instructed to argue its assigned side persuasively but honestly (no strawmanning), and to directly rebut the opponent's most recent point.
3. Run 3 rounds of alternating turns via a `StateGraph`, where each debater sees the full transcript so far.
4. A neutral **Judge** agent reviews the full transcript and produces a structured (Pydantic) verdict: strongest point from each side, and a final synthesized recommendation.
5. Print the round-by-round transcript, the judge's verdict, and compare the judge's nuanced answer to what either debater alone concluded.

In [ ]:
# ============ DEBATE QUESTION & PERSONAS ============
DEBATE_QUESTION = (
    "Should a 6-person engineering team building a new B2B SaaS product "
    "adopt a microservices architecture or a monolith for their first version?"
)

PRO_MICROSERVICES_PERSONA = """You are the Pro-Microservices Advocate in a structured debate.
You must argue IN FAVOR of adopting a microservices architecture for the question at hand.

Rules:
- Argue persuasively but honestly. Do not invent facts or strawman the opposing side.
- If this is not your opening turn, you MUST directly rebut the single strongest point your
  opponent made in their most recent turn before advancing your own argument.
- Keep your turn focused: one clear rebuttal (if applicable) + one or two new supporting points.
- Keep responses to at most 120 words.
- Never break character or acknowledge you are an AI."""

PRO_MONOLITH_PERSONA = """You are the Pro-Monolith Advocate in a structured debate.
You must argue IN FAVOR of adopting a monolithic architecture for the question at hand.

Rules:
- Argue persuasively but honestly. Do not invent facts or strawman the opposing side.
- If this is not your opening turn, you MUST directly rebut the single strongest point your
  opponent made in their most recent turn before advancing your own argument.
- Keep your turn focused: one clear rebuttal (if applicable) + one or two new supporting points.
- Keep responses to at most 120 words.
- Never break character or acknowledge you are an AI."""

MAX_ROUNDS = 3

In [ ]:
# ============ STATE DEFINITION ============
class Turn(TypedDict):
    round: int
    speaker: str
    content: str


class DebateState(TypedDict):
    question: str
    transcript: List[Turn]
    round_number: int
    max_rounds: int
    verdict: dict

In [ ]:
# ============ HELPER: RENDER TRANSCRIPT FOR A PROMPT ============
def render_transcript(transcript: List[Turn]) -> str:
    if not transcript:
        return "(No arguments have been made yet. You are opening the debate.)"
    lines = [f"[Round {t['round']}] {t['speaker']}: {t['content']}" for t in transcript]
    return "\n\n".join(lines)

In [ ]:
# ============ DEBATER NODES ============
def make_debater_node(persona_prompt: str, speaker_name: str):
    """Factory that builds a debater node bound to a fixed persona/stance."""

    def debater_node(state: DebateState) -> DebateState:
        transcript_so_far = render_transcript(state["transcript"])
        human_prompt = (
            f"Debate question: {state['question']}\n\n"
            f"Transcript so far:\n{transcript_so_far}\n\n"
            "Give your next turn now."
        )
        response = llm.invoke(
            [SystemMessage(content=persona_prompt), HumanMessage(content=human_prompt)]
        )
        new_turn: Turn = {
            "round": state["round_number"],
            "speaker": speaker_name,
            "content": response.content,
        }
        state["transcript"] = state["transcript"] + [new_turn]
        return state

    return debater_node


pro_microservices_node = make_debater_node(
    PRO_MICROSERVICES_PERSONA, "Pro-Microservices Advocate"
)
pro_monolith_node = make_debater_node(PRO_MONOLITH_PERSONA, "Pro-Monolith Advocate")


def advance_round(state: DebateState) -> DebateState:
    # A "round" completes once both debaters have spoken; increment after Monolith's turn.
    state["round_number"] = state["round_number"] + 1
    return state


def should_continue_debate(state: DebateState) -> Literal["pro_microservices", "judge"]:
    if state["round_number"] > state["max_rounds"]:
        return "judge"
    return "pro_microservices"

In [ ]:
# ============ JUDGE AGENT (STRUCTURED OUTPUT) ============
class DebateVerdict(BaseModel):
    """Structured verdict produced by the neutral Judge after reviewing a full debate transcript."""

    strongest_pro_microservices_point: str = Field(
        description="The single strongest, most defensible point raised by the Pro-Microservices Advocate."
    )
    strongest_pro_monolith_point: str = Field(
        description="The single strongest, most defensible point raised by the Pro-Monolith Advocate."
    )
    weaknesses_identified: str = Field(
        description="Brief note on any weak, unsupported, or overstated claims made by either side."
    )
    final_recommendation: str = Field(
        description=(
            "A final, nuanced, synthesized recommendation that may be more conditional/balanced "
            "than either single debater's position (e.g. 'it depends on X, Y' or a staged approach)."
        )
    )
    confidence: str = Field(
        description="Judge's confidence in the recommendation: 'low', 'medium', or 'high'."
    )


judge_llm = llm.with_structured_output(DebateVerdict)

JUDGE_SYSTEM_PROMPT = """You are a neutral, impartial Judge reviewing a structured debate.
You have no stake in either side's position. Your job is to:
1. Identify the single strongest point each side made.
2. Note any weak or overstated claims from either side.
3. Produce a final recommendation that is allowed to be MORE nuanced than either debater's
   position -- for example, a conditional answer, a staged/phased approach, or a synthesis
   that borrows the best ideas from both sides.
Do not simply declare one side the 'winner' -- your job is to reason past the debate, not referee it."""


def judge_node(state: DebateState) -> DebateState:
    transcript_so_far = render_transcript(state["transcript"])
    human_prompt = (
        f"Debate question: {state['question']}\n\n"
        f"Full transcript:\n{transcript_so_far}\n\n"
        "Produce your structured verdict now."
    )
    verdict: DebateVerdict = judge_llm.invoke(
        [SystemMessage(content=JUDGE_SYSTEM_PROMPT), HumanMessage(content=human_prompt)]
    )
    state["verdict"] = verdict.model_dump()
    return state

In [ ]:
# ============ BUILD THE DEBATE GRAPH ============
debate_graph = StateGraph(DebateState)

debate_graph.add_node("pro_microservices", pro_microservices_node)
debate_graph.add_node("pro_monolith", pro_monolith_node)
debate_graph.add_node("advance_round", advance_round)
debate_graph.add_node("judge", judge_node)

debate_graph.add_edge(START, "pro_microservices")
debate_graph.add_edge("pro_microservices", "pro_monolith")
debate_graph.add_edge("pro_monolith", "advance_round")
debate_graph.add_conditional_edges(
    "advance_round",
    should_continue_debate,
    {"pro_microservices": "pro_microservices", "judge": "judge"},
)
debate_graph.add_edge("judge", END)

debate_app = debate_graph.compile()

In [ ]:
# ============ VISUALIZE THE GRAPH ============
from IPython.display import Image, display
from langchain_core.runnables.graph import MermaidDrawMethod

display(
    Image(
        debate_app.get_graph().draw_mermaid_png(
            draw_method=MermaidDrawMethod.API,
        )
    )
)

In [ ]:
# ============ RUN THE DEBATE ============
initial_state: DebateState = {
    "question": DEBATE_QUESTION,
    "transcript": [],
    "round_number": 1,
    "max_rounds": MAX_ROUNDS,
    "verdict": {},
}

result = debate_app.invoke(initial_state)

In [ ]:
# ============ PRINT THE FULL TRANSCRIPT ============
print(f"DEBATE QUESTION:\n{result['question']}\n")
print("=" * 80)
for turn in result["transcript"]:
    print(f"\n--- Round {turn['round']} | {turn['speaker']} ---")
    print(turn["content"])
print("\n" + "=" * 80)

In [ ]:
# ============ PRINT THE JUDGE'S VERDICT ============
verdict = result["verdict"]
print("JUDGE'S VERDICT\n" + "-" * 80)
print(f"Strongest Pro-Microservices point:\n  {verdict['strongest_pro_microservices_point']}\n")
print(f"Strongest Pro-Monolith point:\n  {verdict['strongest_pro_monolith_point']}\n")
print(f"Weaknesses identified:\n  {verdict['weaknesses_identified']}\n")
print(f"Final synthesized recommendation:\n  {verdict['final_recommendation']}\n")
print(f"Judge confidence: {verdict['confidence']}")

## Discussion of the Output

Run the cells above and compare three things side by side:

1. **The Pro-Microservices Advocate's own conclusion** — taken alone, it would push the team toward
   microservices unconditionally, because that is the only side of the trade-off it was assigned to see.
2. **The Pro-Monolith Advocate's own conclusion** — symmetrically, it would push toward a monolith
   unconditionally, for the same reason.
3. **The Judge's `final_recommendation`** — because the judge saw both fully-argued, rebutted positions,
   its answer is typically *conditional* rather than absolute: something like "start with a well-modularized
   monolith with clear internal boundaries, and only peel off services once specific scaling or team-topology
   pressures appear" — a position neither single debater would have reached alone, because neither was
   incentivized to concede ground to the other side.

This is the core value of the Debate pattern: the adversarial process surfaces the strongest version of
*both* arguments before a final decision is made, which tends to produce a more defensible, better-hedged
answer than a single forward-pass "just answer the question" prompt would — at the cost of `2N + 1` LLM
calls instead of one.

## Key Takeaways

- **Debate** assigns opposing personas/stances to two or more agents and lets them argue over multiple
  rounds, each seeing the full transcript so far and directly rebutting the opponent's latest point.
- A **neutral Judge agent**, prompted specifically to synthesize rather than simply pick a "winner",
  produces a structured verdict (via Pydantic / `with_structured_output`) that is typically more nuanced
  and better-reasoned than either single debater's position.
- The control flow is a small **round-robin `StateGraph`**: two debater nodes alternate, a round counter
  drives a conditional edge, and once `max_rounds` is reached the graph routes to the judge node and ends.
- The pattern trades extra LLM calls (`2N + 1` for `N` rounds) for a more defensible, auditable answer on
  genuinely debatable questions — it is not worth the cost on simple factual questions with one correct answer.
- Persona/system-prompt design matters a lot: both debaters must be instructed to argue honestly and to
  engage with (not ignore) the opponent's strongest point, or the debate degenerates into two parallel
  monologues with nothing for the judge to actually adjudicate.